# Install

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install openai tenacity func_timeout rapidfuzz sqlglot pandas tqdm

# Load Model

In [ ]:
import os
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

API_KEY = os.environ["QWEN_API_KEY"]
BASE_URL = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen3.5-plus"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Load Dataset

In [ ]:
DB_ROOT_PATH = 'path/to/dev_corrected/dev_databases'

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
from huggingface_hub import hf_hub_download
import json
repo_id = "RuihanCao/bird-dev-corrected"

# Load the Indexed Dataset
dataset_path = hf_hub_download(repo_id=repo_id, filename="dev_dataset_indexed.json", repo_type="dataset")
with open(dataset_path, 'r') as f:
    eval_dataset = json.load(f)

# Load the Gold Results
gold_path = hf_hub_download(repo_id=repo_id, filename="gold_results.json", repo_type="dataset")
with open(gold_path, 'r') as f:
    gold_results = json.load(f)

# Load the Schema Cache
schema_path = hf_hub_download(repo_id=repo_id, filename="bird_schema_cache.json", repo_type="dataset")
with open(schema_path, 'r') as f:
    schema_cache = json.load(f)

print(f"✅ Loaded {len(eval_dataset)} entries and {len(schema_cache)} cached schemas.")

In [ ]:
print(eval_dataset[0])
print(gold_results['0'])
print(schema_cache['california_schools'])

# Helper Functions

In [ ]:
import time
from func_timeout import func_timeout, FunctionTimedOut

class MockCompletionOutput:
    def __init__(self, text):
        self.text = text

class MockRequestOutput:
    def __init__(self, texts):
        # Mimics vLLM's list of outputs for a single prompt
        self.outputs = [MockCompletionOutput(t) for t in texts]

def _run_query(db_path, sql):
      conn = sqlite3.connect(db_path)
      cursor = conn.cursor()
      cursor.execute(sql)
      result = cursor.fetchall()
      conn.close()
      return set(result)

def execute_sql(db_path, sql, timeout_s=10, max_rows=20000):
    conn = sqlite3.connect(db_path)
    start = time.time()

    def progress():
        # return 1 to abort the query
        return 1 if (time.time() - start) > timeout_s else 0

    # call progress() every N VM steps
    conn.set_progress_handler(progress, 10000)

    try:
        cur = conn.cursor()
        cur.execute(sql)

        rows = []
        while True:
            chunk = cur.fetchmany(2000)
            if not chunk:
                break
            rows.extend(chunk)
            if len(rows) > max_rows:
                return "Error: Too many rows"
        return set(rows)

    except sqlite3.OperationalError as e:
        if "interrupted" in str(e).lower():
            return "Error: Timeout (SQL took too long)"
        return f"Error: {e}"
    finally:
        conn.close()

def clean_sql(text, word_limit=375):
    # Remove the <think> block
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # Remove <answer> tags
    text = re.sub(r'<answer>', '', text, flags=re.DOTALL)
    text = re.sub(r'</answer>', '', text, flags=re.DOTALL)
    # Extract code block if it exists (Standard Instruct format)
    # Looks for ```sql ... ``` or just ``` ... ```
    code_block = re.search(r'```(?:sql)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if code_block:
        return code_block.group(1).strip()

    # Fallback
    extracted = text.strip()
    words = extracted.split()
    if len(words) > word_limit:
        #print(f"⚠️ Extraction Warning: SQL candidate too long ({len(words)} words). Truncating to last {word_limit} words.")
        # Take the last 375 words
        extracted = " ".join(words[-word_limit:])
    return extracted

def format_result_log(res, max_len=100):
    if isinstance(res, str):
        return res # Return full error message

    res_str = str(res)
    if len(res_str) > max_len:
        return res_str[:max_len] + "... [TRUNCATED]"
    return res_str

import sqlite3
import sqlglot
from sqlglot import exp
from rapidfuzz import process, fuzz
import os

def fix_sql_values_fuzzy(db_path, sql_query, threshold=85):
    """
    Parses SQL, extracts string literals in WHERE/HAVING clauses,
    and fuzzy matches them against the database content to fix typos/case issues.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database not found at: {db_path}")

    # Connect to DB
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Parse SQL
    try:
        parsed = sqlglot.parse_one(sql_query, read="sqlite")
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return sql_query  # Return original if parse fails

    # Identify tables in the query to limit search scope
    tables = [t.name for t in parsed.find_all(exp.Table)]

    # Extract binary operations involving strings (e.g., col = 'string', col LIKE 'string')
    # We look for Equality (EQ) or Like (LIKE) nodes where one side is a Literal String
    modifications = {}

    for node in parsed.find_all(exp.EQ):
        left = node.left
        right = node.right

        # Identify which side is the column and which is the string literal
        col_node = None
        val_node = None

        if isinstance(left, exp.Column) and isinstance(right, exp.Literal) and right.is_string:
            col_node = left
            val_node = right
        elif isinstance(right, exp.Column) and isinstance(left, exp.Literal) and left.is_string:
            col_node = right
            val_node = left

        if col_node and val_node:
            original_value = val_node.this
            column_name = col_node.name

            # 1. Try to find the exact column in the DB to fetch valid values
            # (Simple resolution: check if column exists in any of the active tables)
            candidate_values = []
            for table in tables:
                try:
                    # Check if column exists in this table
                    cursor.execute(f"PRAGMA table_info({table})")
                    columns_info = cursor.fetchall() # list of tuples
                    col_names = [info[1] for info in columns_info]

                    if column_name in col_names:
                        # Fetch distinct values from this column
                        # LIMITing to avoid massive overhead on huge tables
                        query = f"SELECT DISTINCT {column_name} FROM {table} WHERE {column_name} IS NOT NULL LIMIT 2000"
                        cursor.execute(query)
                        results = cursor.fetchall()
                        candidate_values.extend([str(r[0]) for r in results])
                except Exception as e:
                    continue # Table might be aliased or complex, skip safe

            if not candidate_values:
                continue # Couldn't find valid values for this column, skip

            # 2. Perform Fuzzy Matching
            # extractOne returns (best_match, score, index)
            best_match = process.extractOne(
                original_value,
                candidate_values,
                scorer=fuzz.ratio, # Use ratio for strictness, or token_sort_ratio for flexibility
                score_cutoff=threshold
            )

            if best_match:
                corrected_value, score, _ = best_match
                if corrected_value != original_value:
                    # Store modification to apply later (or modify in place)
                    # We modify in place using sqlglot
                    print(f"Fixing: '{original_value}' -> '{corrected_value}' (Score: {score})")
                    val_node.set("this", corrected_value)

    conn.close()

    # Return the reconstructed SQL
    return parsed.sql(dialect="sqlite")


from dataclasses import dataclass
import math
from typing import Any, Dict, List

@dataclass(frozen=True)
class ComparableTuple:
    """
    A tuple wrapper that handles float comparison with tolerance (1e-9).
    Crucial for voting on results where 10.0000001 must equal 10.0.
    """
    data: tuple

    def __eq__(self, other):
        if not isinstance(other, ComparableTuple):
            return False
        if len(self.data) != len(other.data):
            return False

        for v1, v2 in zip(self.data, other.data):
            if isinstance(v1, float) and isinstance(v2, float):
                if not math.isclose(v1, v2, rel_tol=1e-9):
                    return False
            elif v1 != v2:
                return False
        return True

    def __hash__(self):
        # Convert floats to strings with limited precision for hashing compatibility
        processed = tuple(f"{x:.8f}" if isinstance(x, float) else x for x in self.data)
        return hash(processed)

def execution_match(prediction: Any, ground_truth: Any) -> bool:
    """
    Compare two execution results for equality, handling float values and row order.
    Adapted to handle both List[Dict] (original code) and List[Tuple] (SQLite output).
    """
    # 1. Handle Error Strings immediately
    if isinstance(prediction, str) or isinstance(ground_truth, str):
        return prediction == ground_truth

    # 2. Handle None/Empty
    if not prediction and not ground_truth: return True
    if not prediction or not ground_truth: return False

    def transform_results(ex_result):
        new_ex_result = []
        for row in ex_result:
            # ADAPTER: Handle SQLite tuples vs Dicts
            if isinstance(row, dict):
                row_vals = tuple(row.values())
            else:
                row_vals = tuple(row)

            new_row = ComparableTuple(row_vals)
            new_ex_result.append(new_row)
        return set(new_ex_result)

    return transform_results(ground_truth) == transform_results(prediction)

def get_votes(candidate_predictions: List[Dict]) -> Dict:
    """
    Count votes for each unique prediction based on execution results.
    Clustering Algorithm: O(N^2) comparison to group semantically identical results.
    """
    clusters = [] # List of tuples: (representative_result, vote_count, [candidate_dicts])

    for candidate in candidate_predictions:
        res = candidate["pred_result"]
        found = False

        # Try to find an existing cluster that matches this result
        for i, (cluster_res, count, members) in enumerate(clusters):
            if execution_match(res, cluster_res):
                clusters[i] = (cluster_res, count + 1, members + [candidate])
                found = True
                break

        # If no match, start a new cluster
        if not found:
            clusters.append((res, 1, [candidate]))
    # Sort clusters by vote count descending
    clusters.sort(key=lambda x: x[1], reverse=True)
    # The winner is the first candidate of the largest cluster
    winner = clusters[0][2][0]
    # Summary of voting for logs
    vote_summary = f"Votes: {', '.join([str(c[1]) for c in clusters])}"
    return {
        "winner": winner,
        "vote_summary": vote_summary
    }

# Generate Prompt Function

In [ ]:
def omni_style_prompt(sample):
    db_id = sample['db_id']
    schema = schema_cache.get(db_id, "")
    evidence = sample['evidence']
    question = sample['question']
    user_prompt = f"""Task Overview:
You are a data science expert. Below, you are provided with a database schema and a natural language question. Your task is to understand the schema and generate a valid SQL query to answer the question.

Database Engine:
SQLite

Database Schema:
{schema}
This schema describes the database's structure, including tables, columns, primary keys, foreign keys, and any relevant relationships or constraints.

Question:
{evidence}
{question}

Instructions:
- Strictly follow the instructions in the question, and do not add filters based on common sense if they are not explicitly mentioned in the question or the evidence.
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
In your answer, please enclose the generated SQL query in a code block:
```
-- Your SQL query
```
"""
    return [
        {"role": "system", "content": "You are a helpful SQL expert. You output only valid SQL wrapped in markdown blocks."},
        {"role": "user", "content": user_prompt}
    ]


# Testing

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
def call_api_single_row(messages, n=1, temperature=0.6):
    """Calls the API for a single problem."""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            n=n, # Request n samples (e.g. 8 for voting)
            temperature=temperature,
            top_p=0.95, # Qwen 3.5 Thinking Mode recommended setting
            max_tokens=4096
        )
        return [choice.message.content for choice in response.choices]
    except Exception as e:
        print(f"API Error: {e}")
        return ["Error"] * n

def batch_generate_api(prompts_list, n=1, temperature=0.6, max_workers=10):
    """
    Simulates model.fast_generate but uses API.
    Returns a list of MockRequestOutput objects to match vLLM structure.
    """
    results = [None] * len(prompts_list)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create a map of future -> index
        future_to_idx = {
            executor.submit(call_api_single_row, prompt, n, temperature): i
            for i, prompt in enumerate(prompts_list)
        }

        for future in tqdm(as_completed(future_to_idx), total=len(prompts_list), desc="API Requests"):
            idx = future_to_idx[future]
            try:
                texts = future.result()
                results[idx] = MockRequestOutput(texts)
            except Exception as e:
                print(f"Failed index {idx}: {e}")
                results[idx] = MockRequestOutput(["Error"])

    return results

In [ ]:
import pandas as pd
import datetime
import re

#Config
START_INDEX = 42
N_SAMPLES = 1  # No n-rep, just 1 generation
#MODEL_NAME = "Qwen3.5-397B-A17B"

# Prepare Prompts & Slice Dataset
# We create tuples of (original_index, sample) to ensure we map back to the correct Gold Result
sliced_data = [(i, sample) for i, sample in enumerate(eval_dataset) if i >= START_INDEX]

# limit it to just 10 items for a quick test
# sliced_data = sliced_data[:10]

prompts = [omni_style_prompt(sample) for _, sample in sliced_data]

print(f"🚀 Starting API Inference (n={N_SAMPLES}) starting from index {START_INDEX}...")
print(f"Total queries to run: {len(prompts)}")

start_gen = time.time()

# Call API (n=1)
outputs = batch_generate_api(
    prompts,
    n=N_SAMPLES,
    temperature = 0.6,
    max_workers=20
)

end_gen = time.time()
print(f"✅ Inference Done! Time: {end_gen - start_gen:.2f}s")

# Execution (Linear pass, no voting, no correction)
final_results = []
result_save_path ="result.csv"

# Iterate over outputs and the corresponding original data
for (original_idx, item), output_group in tqdm(zip(sliced_data, outputs), total=len(outputs), desc="Executing SQLs"):

    db_path = os.path.join(DB_ROOT_PATH, item['db_id'], f"{item['db_id']}.sqlite")

    # Extract the single result (since n=1)
    if output_group.outputs:
        raw_text = output_group.outputs[0].text
    else:
        raw_text = "Error"

    # Parse
    parsed_sql = clean_sql(raw_text)

    pred_sql = parsed_sql
    # Execute
    pred_result = execute_sql(db_path, pred_sql)

    is_error = isinstance(pred_result, str)
    is_empty = (not is_error) and (len(pred_result) == 0)

    # Compare with Gold Logic
    # use str(original_idx) to fetch the correct gold answer key
    gold_data = gold_results.get(str(original_idx), {})
    gold_json_str = gold_data.get('gold_result', "[]")

    try:
        gold_list = json.loads(gold_json_str)
        # Convert lists to tuples for set comparison
        gold_set = set(tuple(x) for x in gold_list)
    except:
        gold_set = set()

    is_correct = execution_match(pred_result, gold_set)

    # Log Data
    final_results.append({
        'Index': original_idx,
        "Question ID": item['question_id'],
        "Question": item['question'],
        "Evidence": item['evidence'],
        "Gold SQL": item['SQL'],
        "Pred SQL": pred_sql,
        "Correct": is_correct,
        "Pred Res": format_result_log(pred_result),
        "Gold Res": format_result_log(gold_set),
        "Difficulty": item['difficulty'],
        "DB_ID": item['db_id'],
        "Gen_Text": raw_text
    })



In [ ]:
# Save to CSV
df_results = pd.DataFrame(final_results)
df_results.to_csv(result_save_path, index=False)
print(f"\n💾 Results saved to: {result_save_path}")

# Calculate Metrics
if not df_results.empty:
    print("\n" + "="*50)
    print("📊 BIRD Benchmark Statistics")
    print("="*50)

    # Total Accuracy
    total_acc = df_results['Correct'].mean() * 100
    print(f"\n✨ Total Accuracy: {total_acc:.2f}%")

    # Difficulty Breakdown
    print("\n[ Accuracy by Difficulty ]")
    print(df_results.groupby('Difficulty')['Correct'].agg(['mean', 'count']))
    print("="*50)
else:
    print("⚠️ No results generated.")